# 1 · Equations of motion — what kind of plant is this?

A onewheel looks like one vehicle and behaves like two, and the thing that decides which is a single number.

**On the data.** Everything plotted here is loaded from `sim/out/experiments/`, produced by `scripts/analyse_control.py` against the same MuJoCo model the controller runs against. The scenarios are deterministic and seeded, so re-running a configuration reproduces it bit-for-bit — these are the numbers the design decisions were actually made on, archived after the fact rather than captured live. Nothing here is hand-typed.

---

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = REPO / "sim" / "out" / "experiments"
sys.path.insert(0, str(REPO))

if not (DATA / "control-analysis.json").exists():
    raise SystemExit("Run:  scripts/analyse_control.py   (it writes the datasets this notebook reads)")

C = json.loads((DATA / "control-analysis.json").read_text())
A = np.load(DATA / "control-analysis.npz")
INK, AMBER, MINT, MUTED = "#16232E", "#F2A24A", "#2AAE97", "#96A8B0"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.25})
print(C["note"])

## The one number: $mgl$

Linearised about upright, pitch obeys

$$I_p\,\ddot{\theta} + b\,\dot{\theta} - mgl\,\theta = k_t\, i$$

where $l$ is the height of the centre of mass **above** the axle. The sign of $mgl$ decides everything:

- $l < 0$ (CoM below the axle) → the $-mgl\theta$ term is *restoring*. A hanging pendulum. It cannot fall over.
- $l > 0$ (CoM above) → the term is *destabilising*. An inverted pendulum. It falls unless actively held.

The driverless board sits in the first case and a ridden one in the second. **Same vehicle, opposite plant.**

In [ ]:
rows = C["plant_sweep"]["rows"]
print(f"{'ballast':>9}{'total':>9}{'CoM-axle':>11}{'mgl':>10}{'tau@5deg':>11}   character")
for r in rows:
    kind = "INVERTED pendulum" if r["inverted"] else "stable pendulum"
    print(f"{r['ballast_kg']:8.0f}kg{r['total_kg']:8.1f}kg{r['com_mm']:+10.0f}mm"
          f"{r['mgl']:+9.1f}{r['tau_5deg_nm']:10.2f}Nm   {kind}")

m = [r["ballast_kg"] for r in rows]
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.6))
a.axhline(0, color=INK, lw=1)
a.plot(m, [r["mgl"] for r in rows], "o-", color=AMBER)
a.set_xlabel("ballast (kg)"); a.set_ylabel(r"$mgl$  (N·m/rad)")
a.set_title("gravity flips from helping to fighting")
b.semilogy(m, [max(r["tau_5deg_nm"], 1e-3) for r in rows], "o-", color=MINT)
b.set_xlabel("ballast (kg)"); b.set_ylabel("N·m")
b.set_title(r"torque to hold 5°:  $\tau = mgl\,\sin\theta$")
plt.tight_layout()

That right-hand plot is log-scaled because the range is otherwise unreadable: **0.2 N·m driverless, 44.7 N·m ridden**. Roughly 200×.

It is also the answer to a question we got wrong early: *what sizes the motor?* Not balancing — a rider-scale board rejects a firm shove on about 15 N·m. It is **acceleration and hill climbing**, at roughly $12\ \mathrm{N\cdot m}$ per $\mathrm{m/s^2}$ for this mass and wheel radius.

---
## Debugging story 1 — the free-swing period, wrong twice

Release the frame from an offset and it swings as a pendulum. One number pins the inertia-to-CoM-offset ratio, which is the parameter pair a model is most likely to have wrong:

$$T = 2\pi\sqrt{\frac{I_p}{mgl}}, \qquad I_p = I_{cm} + m l^2$$

The design doc first said **2.1 s**. That used the *whole board's* 12.5 kg — but the wheel is centred on the axle and spins freely, so it contributes neither restoring torque nor pitch inertia. Frame-only gives **2.61 s**. The doc had even carried a warning against exactly that mistake, three lines above the number that made it.

Then a review pointed out 2.61 s is the *wheel-decoupled limit*. The hinge has damping, so the wheel is dragged along — a lightly-coupled absorber — and the true period sits somewhere between free and locked.

In [ ]:
f = C["freeswing"]
t, th = A["freeswing_t"], np.degrees(A["freeswing_pitch_rad"])

plt.figure(figsize=(10, 3.4))
plt.plot(t, th, color=INK, lw=1.4)
for label, key, col in (("whole-board (the original error)", "analytic_whole_board_s_THE_ORIGINAL_ERROR", AMBER),
                        ("frame-only", "analytic_frame_only_s", MUTED),
                        ("wheel locked", "analytic_wheel_locked_s", MINT)):
    plt.axvline(f[key], ls="--", lw=1.3, color=col, label=f"{label}: {f[key]:.2f} s")
plt.axvline(f["measured_s"], lw=2.0, color="crimson", label=f"MEASURED: {f['measured_s']:.2f} s")
plt.xlim(0, 4); plt.xlabel("time (s)"); plt.ylabel("pitch (deg)")
plt.title("free swing: one period, three predictions, one measurement")
plt.legend(fontsize=8); plt.tight_layout()

print(f"predicted band [frame-only, wheel-locked] = "
      f"[{f['analytic_frame_only_s']:.2f}, {f['analytic_wheel_locked_s']:.2f}] s")
print(f"measured                                  =  {f['measured_s']:.2f} s")

The measurement lands at the **wheel-locked** end of the band rather than in the middle. Worth being honest about why: in this measurement the axle is pinned by forcing its position every step, which also constrains how freely the wheel can react. So the number is real but the *setup* is closer to "wheel locked" than a true free swing would be.

**The lesson is the band, not the point.** A single predicted value would have been asserted and believed; a band derived from two bounding assumptions immediately shows which assumption the measurement actually matches.

---
## Debugging story 2 — a false pivot that predicted the wrong answer

An early version of the design doc said the free board pitches about its **contact patch**. That sounds reasonable and is wrong, and it is self-refuting: the CoM sits 0.115 m *above* the contact patch, so that model predicts an inverted pendulum — while the same table said the board was stable, and the impulse test asserts it does not topple.

The board is stable because **the wheel rolls freely and cannot supply the constraint torque** a real pivot would. The correct 2-DOF reduction gives an effective pitch inertia of $C - B^2/A = 0.4034\ \mathrm{kg\,m^2}$ against $0.4072$ pinned — **within 1%**.

So the real difference between a pinned test stand and a free board is not stability at all. It is that **pinning removes translation** — and translation is the mechanism that makes balancing hard.

---
## Debugging story 3 — amps written into a newton-metre channel

The MuJoCo actuator is a `motor` with `gear="1"`, so `data.ctrl` is **torque in N·m**. The scenario commanded **current in amps**, and nothing converted between them.

That baked in an implicit $k_t = 1.0\ \mathrm{N\,m/A}$ and — worse — made `ctrlrange` (N·m) and `max_current_a` (A) *numerically coincide at 40*, so the error was invisible. Every torque-headroom claim in the design docs was comparing two different units, including a confident "50× headroom, saturation cannot bind".

$$\tau = k_t\, i \qquad k_t = 0.7\ \mathrm{N\,m/A}\ \ \textbf{(UNFITTED — the first thing the bench must measure)}$$

**Torque figures in this project are robust; current figures scale with $k_t$.** Read the torque column.

---
## What to take away

1. $mgl$ and its **sign** are the plant. Everything else follows.
2. The driverless board is a *development fixture*, not a small version of the product — it differs qualitatively, not quantitatively.
3. Prefer a **band** to a point estimate. A band tells you which assumption your measurement matched; a point estimate just tells you that you were roughly right or roughly wrong.
4. Name your units in the identifier. `KT_NM_PER_A` cannot be silently confused with amps; `ctrlrange` can.